In [ ]:
import pandas as pd
import mlflow

from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from src.features.engineer import create_preprocessor, FeatureEngineer

import sys
import os
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.append(root_path)


#### **1. Preparing Data**

In [ ]:
df = pd.read_csv("../data/raw/train.csv")

target = "Churn"

X = df.drop(columns=[target])
y = df[target]

In [ ]:
X.head(2)

In [ ]:
y.head(2)

In [ ]:
# train, test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

#### **2. Running Model Pipeline**

In [ ]:
# full pipeline
pipeline = Pipeline([
    ("feature_engineering", FeatureEngineer()),
    ("preprocessor", create_preprocessor()),
    ("model", RandomForestClassifier(random_state=42))
])

# hyperparameter tuning space
param_dist = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [5, 10, 15, None],
    "model__min_samples_split": [2, 5, 10]
}


In [ ]:
# mlflow
mlflow.set_experiment("churn_prediction_model")

In [ ]:
search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=10,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)

best_model = search.best_estimator_

#### **3. Evaluation**

In [ ]:
y_pred = best_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print(f"Accuracy: {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 (binary): {f1:.4f}")
print(f"F1 (macro): {f1_macro:.4f}")
print(f"F1 (weighted): {f1_weighted:.4f}")

#### **4. Log MLflow**

In [ ]:
with mlflow.start_run():

    mlflow.log_params(search.best_params_)

    mlflow.log_metrics({
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    })

    mlflow.sklearn.log_model(best_model, "model")